# Old Persian Data Processing Notebook

## Purpose of the Notebook
This notebook demonstrates the preparation of structured datasets from the ARIO Old Persian corpus for potential integration into the Trismegistos database. It extracts and transforms both catalogue metadata and entity-related information from the original JSON files into standardized Excel tables suitable for further analysis, visualization, and database import. The workflow includes the preparation of the catalogue, the extraction of glossary entries and their occurrences in the texts, and the integration of these datasets into a final entity table.

## Catalogue Extraction
This section describes the preparation of the Old Persian catalogue from the ARIO JSON data. Relevant metadata were extracted, standardized, and enriched with additional information to produce a structured catalogue suitable for further analysis and potential integration into Trismegistos.

### Field Extraction and Cleaning
The relevant catalogue fields were selected from the ARIO JSON data and organized into a structured table. The language information was then standardized to retain only the identified languages—Akkadian, Old Persian, Elamite, and Hieroglyphs—and the original unprocessed language field was removed.

In [1]:
import json
import os
import pandas as pd

ario_catalogue = r"data\catalogue.json"

with open(ario_catalogue, "r", encoding="utf-8") as f:
    data = json.load(f)

members = data["members"]

# Columns: Left  = field name in JSON, Right = column name in output Excel
json_columns = {
    "id_composite": "ID",
    "popular_name": "Name",
    "provenience": "Provenience",
    "ruler": "Ruler",
    "language": "Language(s)"
    }

# Convert selected JSON fields to rows
rows = []

for item_id, item in members.items():
    row = {}

    for json_field, column_name in json_columns.items():
        row[column_name] = item.get(json_field, "")

    rows.append(row)

# Create DataFrame
df_cat = pd.DataFrame(rows)

# Languages
def clean_languages(x):
    text = str(x)
    clean = []
    if "Akkadian" in text:
        clean.append("Akkadian")
    if "Old Persian" in text or "Persian" in text:
        clean.append("Old Persian")
    if "Elamite" in text:
        clean.append("Elamite")
    if "Hieroglyphs" in text:
        clean.append("Hieroglyphs")
    return ", ".join(clean)

df_cat["Languages"] = df_cat["Language(s)"].apply(clean_languages)

# Remove original raw language column from final output
df_cat = df_cat.drop(columns=["Language(s)"])

### Metadata Enrichment
Additional metadata were added to enrich the catalogue. Approximate date ranges were assigned based on the recorded ruler, and each provenience was linked to a modern location together with its geographic coordinates to support subsequent mapping and visualization.

The distinct ruler and provenience values in the catalogue were inspected before enriching the dataset. The ruler values were used to assign approximate date ranges, while the provenience values were used to map the historical locations to their modern names and geographic coordinates.

In [2]:
df_cat["Provenience"].unique()

array(['Babylon', 'Ur', 'Uruk', 'Ecbatana', 'Pasargadae', 'Bisutun',
       'Elvend', 'Gherla (Romania)', 'El-Khargeh', 'Naqsh-I Rustam',
       'Persepolis', 'Susa', 'Suez', 'Persia', 'Faqous', 'Tushpa',
       'Daskyleion', 'Unknown', 'Bost'], dtype=object)

In [3]:
df_cat["Ruler"].unique()

array(['Cyrus II', 'Ariaramnes', 'Arsames', 'Darius I', 'Xerxes I',
       'Artaxerxes I', 'Darius II', 'Artaxerxes II',
       'Artaxerxes II or III', 'Artaxerxes III'], dtype=object)

In [4]:
# Add Date column based on Ruler
rulers = {
    "Cyrus II": "560-530 BC",
    "Darius I": "522-486 BC",
    "Xerxes I": "486-465 BC",
    "Artaxerxes I": "465-424 BC",
    "Darius II": "423-405 BC",
    "Artaxerxes II": "405-358 BC",
    "Artaxerxes III": "358-338 BC",
    "Artaxerxes IV": "338-336 BC",
    "Darius III": "336-330 BC",

    # manually choose the overall range from 405-358 BC or 358-338 BC
    "Arsames": "405-338 BC",
    "Ariaramnes": "405-338 BC",
    "Artaxerxes II or III": "405-338 BC"
}

# Add Modern Location, Latitude, and Longitude column based on Location
locations = {
    "Babylon": ("Hillah, Iraq", 32.4721, 44.4200),
    "Ur": ("Tell el-Muqayyar, Iraq", 30.9621, 46.1037),
    "Uruk": ("Warka, Iraq", 31.3242, 45.6376),
    "Ecbatana": ("Hamadan, Iran", 34.7992, 48.5146),
    "Pasargadae": ("Pasargad, Iran", 30.2030, 53.1790),
    "Bisutun": ("Bisotun / Behistun, Iran", 34.3892, 47.4367),
    "Elvend": ("Mount Alvand / Ganjnameh, Hamadan, Iran", 34.7603, 48.4364),
    "Gherla (Romania)": ("Gherla, Romania", 47.0333, 23.9167),
    "El-Khargeh": ("Kharga Oasis, Egypt", 25.4390, 30.5586),
    "Naqsh-I Rustam": ("Naqsh-e Rostam, Iran", 29.9884, 52.8746),
    "Persepolis": ("Takht-e Jamshid, Iran", 29.9350, 52.8916),
    "Susa": ("Shush, Iran", 32.1942, 48.2436),
    "Suez": ("Suez, Egypt", 29.9668, 32.5498),
    "Persia": ("Fars, Iran", 29.1044, 53.0459),
    "Faqous": ("Faqous, Egypt", 30.7303, 31.7970),
    "Tushpa": ("Van, Turkey", 38.5010, 43.3393),
    "Daskyleion": ("Ergili, Turkey", 40.0947, 28.0344),
    "Bost": ("Lashkargah, Afghanistan", 31.5831, 64.3692),
}

### Final Dataset Assembly and Export
The enriched metadata were added to the catalogue, together with a direct ARIO reference link for each record. The final columns were then selected and arranged in a consistent order before the completed catalogue was exported to Excel.

In [5]:
# Add columns Date,  Modern Location, Latitude, Longitude, and Reference
df_cat["Date"] = df_cat["Ruler"].map(rulers).fillna("")

df_cat["Modern Location"] = df_cat["Provenience"].map(locations).str[0].fillna("")

df_cat["Reference"] = "http://oracc.org/ario/" + df_cat["ID"]

df_cat["Index"] = range(1, len(df_cat) + 1)

In [6]:
# Final Catalogue Output
catalogue_columns = [
    "Index",
    "ID",
    "Name",
    "Provenience",
    "Ruler",
    "Date",
    "Languages",
    "Modern Location",
    "Reference"
]

df_catalogue = df_cat[catalogue_columns].copy()

catalogue_output = r"output\Old_Persian_Catalogue.xlsx"

df_catalogue.to_excel(catalogue_output, index=False)

A separate version of the catalogue was also created for Tableau, with additional fields required for the visualisations.

In [7]:
# Tableau Output

df_tableau = df_cat.copy()


# Create one separate column for each language

df_tableau["Akkadian"] = df_tableau["Languages"].apply(
    lambda x: "Akkadian" if "Akkadian" in x else ""
)

df_tableau["Old Persian"] = df_tableau["Languages"].apply(
    lambda x: "Old Persian" if "Old Persian" in x else ""
)

df_tableau["Elamite"] = df_tableau["Languages"].apply(
    lambda x: "Elamite" if "Elamite" in x else ""
)

df_tableau["Hieroglyphs"] = df_tableau["Languages"].apply(
    lambda x: "Hieroglyphs" if "Hieroglyphs" in x else ""
)


# Create numerical date fields for Tableau

dates = df_tableau["Date"].str.replace(" BC", "", regex=False)

df_tableau[["Start", "End"]] = dates.str.split("-", expand=True)

df_tableau["Start"] = -pd.to_numeric(
    df_tableau["Start"], errors="coerce"
)

df_tableau["End"] = -pd.to_numeric(
    df_tableau["End"], errors="coerce"
)

df_tableau["Duration"] = (
    df_tableau["End"] - df_tableau["Start"]
)


# Set Tableau column order

tableau_columns = [
    "Index",
    "ID",
    "Name",
    "Provenience",
    "Ruler",
    "Date",
    "Start",
    "End",
    "Duration",
    "Languages",
    "Akkadian",
    "Old Persian",
    "Elamite",
    "Hieroglyphs",
    "Modern Location",
    "Reference"
]

df_tableau = df_tableau[tableau_columns]

tableau_output = r"output\Old_Persian_Catalogue_Tableau.xlsx"

df_tableau.to_excel(tableau_output, index=False)

## Entity Extraction

This section describes how the entity-related data from the ARIO files was prepared for further use. The process included extracting glossary entries, identifying their occurrences in the texts, and combining both datasets into a single structured table.

**Note**: The JSON files obtained from the ARIO project also contain some entities from other languages, particularly Akkadian. However, this material appears to be incomplete: the number of entries is relatively small, and some entries lack glosses, although their word class is still provided. Therefore, this dataset has been restricted to Old Persian only. As a result, it represents **Old Persian entities in Achaemenid inscriptions**, rather than **all entities occurring in Achaemenid inscriptions**, which were also written in other languages such as Elamite, Akkadian, and Egyptian.

### Glossary Preparation
The glossary data were extracted from the ARIO JSON file and organized into a structured table containing the glossary ID, ARIO occurrence count, word form, meaning, and part-of-speech category. The POS codes were also mapped to descriptive entity classes, such as person, place, deity, ruler, and river. The resulting Old Persian glossary table was then exported to Excel.

In [8]:
# Input JSON catalogue from ARIO
ario_gloss = r"data\gloss-qpn.json"

columns = {
    "id": "Glossary ID",
    "icount": "ARIO Count",
    "cf": "Word",
    "gw": "Meaning",
    "pos": "POS",
}

# Read JSON file
with open(ario_gloss, "r", encoding="utf-8") as f:
    data = json.load(f)

df_gloss = pd.DataFrame(data["entries"])

df_gloss = df_gloss[list(columns.keys())]
df_gloss = df_gloss.rename(columns=columns)

pos_class = {
    "DN": "Deity",
    "EN": "Ethnicity",
    "GN": "Place",
    "MN": "Month",
    "PN": "Person",
    "RN": "Ruler",
    "SN": "State",
    "WN": "River",
    "": ""
}

df_gloss["Word Class"] = df_gloss["POS"].map(pos_class).fillna("")

# Output Excel file for TM
gloss_excel = r"output\Old_Persian_Glossary.xlsx"

df_gloss.to_excel(gloss_excel, index=False)

print("Occurrence rows:", len(df_gloss))

Occurrence rows: 202


#### Identifying Relevant POS Categories
The POS codes present in the entity glossary were first identified. Because the glossary already contains only entity entries, these codes were subsequently used as selection criteria to extract the corresponding occurrences from the full Old Persian corpus.

In [9]:
print(df_gloss["POS"].unique())

['DN' 'SN' 'MN' 'PN' 'EN' 'GN' 'RN' 'TN' 'LN' 'WN' 'ON' 'QN']


### Occurrences
The occurrence data were extracted from the Old Persian corpus JSON files to identify where selected entity-related words appeared in the texts. Each row represented one occurrence and included its text reference, language, word form, and part-of-speech category. The data were then limited to Old Persian entries in the selected categories, and an occurrence count was added for each word and POS combination.

In [10]:
# Input folder containing ARIO corpus JSON files
input_folder = r"data\corpusjson"

# Output Excel file
output_excel = r"output\Old_Persian_Occurrences.xlsx"

rows = []

# Read JSON files
for file_name in os.listdir(input_folder):
    file_path = os.path.join(input_folder, file_name)

    # Skip empty files
    if os.path.getsize(file_path) == 0:
        continue

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Extract word occurrences
    items = list(data.get("cdl", []))

    while items:
        item = items.pop()

        if not isinstance(item, dict):
            continue

        if item.get("node") == "l":
            word_data = item.get("f", {})

            rows.append({
                "Reference": item.get("ref"),
                "Language": word_data.get("lang"),
                "Word": word_data.get("cf"),
                "POS": word_data.get("pos"),
            })

        if isinstance(item.get("cdl"), list):
            items.extend(item["cdl"])

# Create DataFrame
df_occurrences = pd.DataFrame(rows)

# Keep only Old Persian occurrences
df_occurrences = df_occurrences[
    df_occurrences["Language"] == "peo"
]

# Keep only selected word classes
selected_pos = [
    "DN", "SN", "MN", "PN", "EN", "GN",
    "RN", "TN", "LN", "WN", "ON", "QN"
]

df_occurrences = df_occurrences[
    df_occurrences["POS"].isin(selected_pos)
]

# Count occurrences of each word and POS
df_occurrences["Occurrence Count"] = (
    df_occurrences.groupby(["Word", "POS"])["Word"].transform("size")
)

# Save final Excel file
df_occurrences.to_excel(output_excel, index=False)

print("Occurrence rows:", len(df_occurrences))

Occurrence rows: 1422


### Glossary–Occurrence Integration
Although each glossary entry has a unique ID, these IDs are not used in the occurrence data and therefore could not serve as matching keys. The occurrence table was merged with the glossary using both the word form (cf/Word) and the part-of-speech information as matching criteria. This prevented duplicate matches for words with the same spelling but different meanings and preserved the original number of occurrence records.

In [11]:
# Output file
merged_excel = r"output\Old_Persian_Occurrences_with_Glossary.xlsx"

# left table = occurrences
# right table = glossary
# keys: Word and POS
df_merged = df_occurrences.merge(
    df_gloss,
    how="left",
    left_on=["Word", "POS"],
    right_on=["Word", "POS"],
    suffixes=("_occurrence", "_gloss")
)

# Set final column order
column_order = [
    "Glossary ID",
    "Reference",
    "Language",
    "Word",
    "Meaning",
    "POS",
    "Word Class",
    "ARIO Count",
    "Occurrence Count"
]

df_merged = df_merged[column_order]
# Save final Excel file
df_merged.to_excel(merged_excel, index=False)

print("Occurrence rows:", len(df_occurrences))
print("Merged rows:", len(df_merged))

Occurrence rows: 1422
Merged rows: 1422


## Output Summary

This notebook produces three Excel datasets prepared from the ARIO Old Persian corpus:

- **Old_Persian_Catalogue.xlsx** – catalogue metadata enriched with standardized language information, dates, modern locations, and reference links.
- **Old_Persian_Catalogue_Tableau.xlsx** - a Tableau-ready version of the catalogue with additional fields for visualisation.
- **Old_Persian_Glossary.xlsx** – glossary of Old Persian entity entries with meanings, POS categories, and entity classes.
- **Old_Persian_Occurrences.xlsx** – occurrences of entity-related words in the corpus, including text references and occurrence counts.
- **Old_Persian_Words_with_Gloss.xlsx** – merged dataset combining occurrence records with glossary information for potential import into Trismegistos.